# BiRefNet-portrait — Fine-tuning on Ties Dataset
**Before running:** Runtime → Change runtime type → T4 GPU

Run cells top to bottom. Cell 6 will ask for your HuggingFace token.

Dataset zip structure expected:
```
ties_dataset.zip
  ties_dataset/
    im/   ← tie composited JPGs  (tie_001_main.jpg ...)
    gt/   ← corrected mask PNGs  (tie_001_main.png ...)
```

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q transformers timm einops kornia

In [ ]:
# Cell 3 — Extract dataset
# UPDATE this path if you placed the zip in a subfolder in Drive
DRIVE_ZIP = '/content/drive/MyDrive/ties_dataset.zip'

import zipfile, os, glob
os.makedirs('/content/data', exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall('/content/data')

# Auto-detect im/gt regardless of nesting inside the zip
im_candidates = glob.glob('/content/data/**/im', recursive=True)
gt_candidates = glob.glob('/content/data/**/gt', recursive=True)
assert im_candidates, f'Could not find im/ folder. Extracted: {os.listdir("/content/data")}'
assert gt_candidates, f'Could not find gt/ folder. Extracted: {os.listdir("/content/data")}'
IM_DIR = im_candidates[0]
GT_DIR = gt_candidates[0]
print(f'im/ -> {IM_DIR}')
print(f'gt/ -> {GT_DIR}')

im_files = sorted(os.listdir(IM_DIR))
gt_files = sorted(os.listdir(GT_DIR))
print(f'Images: {len(im_files)}, Masks: {len(gt_files)}')
assert len(im_files) == len(gt_files), 'Mismatch between images and masks count'

In [ ]:
# Cell 4 — Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU — change runtime to T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 5 — Dataset, augmentation, loss
import random
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 512 instead of 1024 — required to fit in T4 VRAM alongside the model
SIZE = 512

class TiesDataset(Dataset):
    def __init__(self, im_dir, gt_dir, augment=True):
        self.im_dir  = Path(im_dir)
        self.gt_dir  = Path(gt_dir)
        self.augment = augment
        self.names   = sorted(p.stem for p in self.im_dir.glob('*.jpg'))

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img  = Image.open(self.im_dir / f'{name}.jpg').convert('RGB')
        mask = Image.open(self.gt_dir / f'{name}.png').convert('L')

        if self.augment:
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.6:
                angle = random.uniform(-10, 10)
                img  = img.rotate(angle, fillcolor=(255, 255, 255))
                mask = mask.rotate(angle, fillcolor=0)
            img = transforms.ColorJitter(
                brightness=0.25, contrast=0.2, saturation=0.15
            )(img)

        img  = img.resize((SIZE, SIZE), Image.LANCZOS)
        mask = mask.resize((SIZE, SIZE), Image.NEAREST)

        img_t  = transforms.Normalize(
            [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        )(transforms.ToTensor()(img))
        mask_t = torch.tensor(
            np.array(mask) / 255.0, dtype=torch.float32
        ).unsqueeze(0)

        return img_t, mask_t


def bce_iou_loss(pred, target):
    # Resize target to match prediction scale (multi-scale supervision)
    if pred.shape[-2:] != target.shape[-2:]:
        target = F.interpolate(target, size=pred.shape[-2:], mode='nearest')
    bce = F.binary_cross_entropy_with_logits(pred, target)
    p   = pred.float().sigmoid()
    t   = target.float()
    inter = (p * t).sum(dim=(2, 3))
    union = (p + t - p * t).sum(dim=(2, 3))
    iou   = 1 - (inter / (union + 1e-8)).mean()
    return bce + iou


dataset = TiesDataset(IM_DIR, GT_DIR, augment=True)
loader  = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
print(f'Dataset ready: {len(dataset)} images at {SIZE}x{SIZE}')

In [ ]:
# Cell 6 — Load BiRefNet (default — works better on ties than portrait)
from getpass import getpass
from transformers import AutoModelForImageSegmentation
import torch

HF_TOKEN = getpass('HuggingFace READ token: ')  # paste token, won't be shown

model = AutoModelForImageSegmentation.from_pretrained(
    'ZhengPeng7/BiRefNet',
    trust_remote_code=True,
    token=HF_TOKEN,
    torch_dtype=torch.float32,
).cuda()

print('Model loaded on', next(model.parameters()).device)
print('dtype:', next(model.parameters()).dtype)

In [ ]:
# Cell 7 — Training
import torch.nn as nn

EPOCHS   = 30
LR       = 1e-5
ACCUM    = 4
SAVE_DIR = '/content/birefnet_ties_finetuned'
DRIVE_OUT = '/content/drive/MyDrive/birefnet_ties_finetuned'

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')
best_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    # BatchNorm must stay in eval mode with batch_size=1 to avoid size=[1,C,1,1] error
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    epoch_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(loader):
        imgs, masks = imgs.cuda(), masks.cuda()

        with torch.amp.autocast('cuda'):
            pred_tensors = model(imgs)[0][1]
            loss = sum(bce_iou_loss(p, masks) for p in pred_tensors) / len(pred_tensors)

        scaler.scale(loss / ACCUM).backward()

        if (step + 1) % ACCUM == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item()

    scheduler.step()
    avg = epoch_loss / len(loader)
    tag = ''
    if avg < best_loss:
        best_loss = avg
        model.save_pretrained(SAVE_DIR)
        tag = '  ← best, saved'
    print(f'Epoch {epoch+1:02d}/{EPOCHS}  loss={avg:.4f}{tag}')

print(f'\nDone. Best loss: {best_loss:.4f}')

# Auto-save to Drive immediately after training — prevents loss if Colab disconnects
import shutil, os
if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(SAVE_DIR, DRIVE_OUT)
print(f'Model saved to Drive: {DRIVE_OUT}')

In [ ]:
# Cell 8 — Save best model to Google Drive (backup — Cell 7 already saves automatically)
DRIVE_OUT = '/content/drive/MyDrive/birefnet_ties_finetuned'
!cp -r {SAVE_DIR} {DRIVE_OUT}
print(f'Saved to Drive: {DRIVE_OUT}')

In [ ]:
# Cell 9 — Visual test: before vs after on holdout images (RAW photos)
# Expects the original raw photos uploaded to MyDrive/:
#   tie_014_main.jpg  (from Products-raw-photos/ties_training/)
#   tie_020_main.jpg  (from Products-raw-photos/ties_training/)
import matplotlib.pyplot as plt
from torchvision import transforms as T

model_orig = AutoModelForImageSegmentation.from_pretrained(
    'ZhengPeng7/BiRefNet', trust_remote_code=True, token=HF_TOKEN,
    torch_dtype=torch.float32,
).cuda().eval()

# Ensure fine-tuned model is float32 (save_pretrained may inherit fp16 config)
model.float().eval()

tf = T.Compose([
    T.Resize((SIZE, SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_names = ['tie_014_main', 'tie_020_main']
fig, axes  = plt.subplots(len(test_names), 3, figsize=(12, 4 * len(test_names)))

for row, name in enumerate(test_names):
    img = Image.open(f'/content/drive/MyDrive/{name}.jpg').convert('RGB')
    inp = tf(img).unsqueeze(0).cuda()

    with torch.no_grad():
        # BiRefNet output: list of predictions — last element is the final (highest-res) mask
        pred_orig  = model_orig(inp)[-1][0].float().sigmoid().squeeze().cpu().numpy()
        pred_tuned = model(inp)[-1][0].float().sigmoid().squeeze().cpu().numpy()

    axes[row][0].imshow(img.resize((512, 512)))
    axes[row][0].set_title(f'{name} — raw photo')
    axes[row][1].imshow(pred_orig,  cmap='gray')
    axes[row][1].set_title('BiRefNet default (before)')
    axes[row][2].imshow(pred_tuned, cmap='gray')
    axes[row][2].set_title('Fine-tuned (after)')
    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ties_finetune_results.png', dpi=80)
plt.show()
print('Results saved to Drive: ties_finetune_results.png')